In [7]:
from calibration_tools import eval_cross_corr_performance, check_violation_range
import numpy as np
from hydra import initialize, compose

In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# $V_{nl,trend}$

In [34]:
fail_rate = []
# we hit the upper limit of the spline samples at 4 so we cant do more extreme. Seems to basically have 0 effect
with initialize(version_base=None, config_path="../../config/"):
    cfg = compose(config_name="generate_dataset.yaml")
performance_stack = []
for x in [12, 10, 8, 6, 4]:
    cfg.nl_mode = "splines"
    cfg.spline_samples = x
    cfg.generator.lagged.nonlinear_proba = 1
    cfg.generator.instant.nonlinear_proba = 1
    res, fails = check_violation_range(cfg)
    performance_stack.append(res)
    fail_rate.append(fails)
print(np.array(performance_stack).round(5))
print(np.array(fail_rate).sum())

[0.96903 0.9718  0.97504 0.97071 0.97265]
0


# $V_{nl,mono}$

In [35]:
# Very little effect but scaling it even higher makes the functions stepfunctions.
with initialize(version_base=None, config_path="../../config/"):
    cfg = compose(config_name="generate_dataset.yaml")
performance_stack = []
power_dist = [
    [0.5, 1, 1, 2],
    [0.25, 0.5, 2, 4],
    [0.125, 0.25, 4, 8],
    [0.0833, 0.125, 8, 12],
    [0.05, 0.0833, 12, 20],
]
for x in power_dist:
    cfg.nl_mode = "power_set"
    cfg.power_dist = x
    cfg.generator.lagged.nonlinear_proba = 1
    cfg.generator.instant.nonlinear_proba = 1
    res, fails = check_violation_range(cfg)
    performance_stack.append(res)
    fail_rate.append(fails)
print(np.array(performance_stack).round(5))
print(np.array(fail_rate).sum())

[0.97128 0.9653  0.9571  0.95135 0.94644]
0


# $V_{nl,rbf}$

In [32]:
# scale up to the max.
with initialize(version_base=None, config_path="../../config/"):
    cfg = compose(config_name="generate_dataset.yaml")
performance_stack = []
for x in [0, 0.425,0.56875,0.7125,0.85625,1]:
    cfg.nl_mode = "rbf"
    cfg.generator.lagged.nl_sampler.limit_startx = [-20.0, 20.0]
    cfg.generator.instant.nl_sampler.limit_startx = [-20.0, 20.0]
    cfg.generator.lagged.nonlinear_proba = x
    cfg.generator.instant.nonlinear_proba = x
    res, fails = check_violation_range(cfg)
    performance_stack.append(res)
    fail_rate.append(fails)
print(np.array(performance_stack).round(5))
print(np.array(fail_rate).sum())

[0.93912 0.90175 0.8646  0.84882 0.803   0.77889]
0


# $V_{nl,comp}$

In [22]:
# Reuse the RBF steps to make it comparable. They also fit reasonably.
with initialize(version_base=None, config_path="../../config/"):
    cfg = compose(config_name="generate_dataset.yaml")
performance_stack = []
for x in [0, 0.05, 0.2875,0.525,0.7625,1]:
    cfg.nl_mode = "symbolic"
    cfg.generator.lagged.nonlinear_proba = x
    cfg.generator.instant.nonlinear_proba = x
    res, fails = check_violation_range(cfg)
    performance_stack.append(res)
    fail_rate.append(fails)
print(np.array(performance_stack).round(5))
print(np.array(fail_rate).sum())

[0.93912 0.90843 0.84714 0.78365 0.73419 0.63739]
0
